# 060_run_pipeline
Runs the whole pipeline: Bronze, Silver, Gold, then publish for Power BI. One batch id for the
whole run. Stops at the first layer with a failure, so a later layer never builds on a broken
one, and nothing is published unless every Gold table succeeded.

On Databricks this would be a Workflow with one task per layer; locally it is this notebook.

In [ ]:
# Bootstrap: find the pipeline root and load the shared notebooks.
# run_notebook() plays the part of Databricks' %run: it runs another notebook's code cells in
# this notebook's namespace. It is used instead of Jupyter's %run because that breaks on
# Windows paths with spaces.
import json
import pathlib

PIPELINE_ROOT = next(
    folder
    for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (folder / "00_shared").is_dir()
)


def run_notebook(relative_path):
    notebook = json.loads((PIPELINE_ROOT / relative_path).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


if "SHARED_LOADED" not in globals():
    run_notebook("00_shared/000_nb_configurations.ipynb")
    run_notebook("00_shared/001_nb_utils.ipynb")
    SHARED_LOADED = True

In [ ]:
# Parameters: True reloads every flow in full. Needed after a change to reference data
# (mapping workbook, FX rates, group lines), because older rows were built with the old values.
# SCD2 dimensions keep their history either way.
full_refresh = False

In [ ]:
pipeline_batch_id = new_batch_id()
pipeline_started = time.time()
print(f"Pipeline batch {pipeline_batch_id}" + ("  (full refresh)" if full_refresh else "") + "\n")

for layer in ["BRONZE", "SILVER", "GOLD"]:
    layer_started = time.time()
    results = run_layer(layer, batch_id=pipeline_batch_id, full_refresh=full_refresh)
    print(f"{layer} took {time.time() - layer_started:,.0f} s\n")
    problems = {cid: r for cid, r in results.items() if r != "SUCCEEDED"}
    if problems:
        raise RuntimeError(
            f"{layer} did not complete ({len(problems)} flow(s) failed or skipped). "
            "Open 05_helper_scripts/052_pipeline_run_report for the details."
        )

## Publish for Power BI

In [ ]:
run_notebook("04_gold/042_publish/0420_publish_to_powerbi.ipynb")
print(f"\nPipeline finished in {time.time() - pipeline_started:,.0f} s. Refresh Power BI.")